# Car Workshop – Run All

Single entry point for the whole pipeline (see `README.md`).
Each `%run` executes the child notebook in this session, so state and widgets are shared.

| section | when to run |
|---|---|
| 1. One-time setup | new environment only (idempotent – safe to re-run) |
| 2. Daily run | every day / backfill |
| 3. Data quality checks | optional |

`TARGET_DATE` widget (`YYYY-MM-DD`, blank = yesterday) is shared with `daily.ipynb`.

In [ ]:
dbutils.widgets.text('TARGET_DATE', '', 'Target date (YYYY-MM-DD, blank = yesterday)')
print(f"TARGET_DATE = {dbutils.widgets.get('TARGET_DATE') or '(yesterday)'}")

## 1. One-time setup

Creates catalog / schemas / volumes / tables, then loads the dimensions.
`create_tables.sql` is all `IF NOT EXISTS` and `initial_dims.ipynb` overwrites
deterministically, so re-running this section is safe – just skip it on daily runs.

In [ ]:
from pathlib import Path

sql_path = Path('create_tables.sql')
assert sql_path.exists(), 'Run this notebook from the car_workshop folder (Databricks Repos)'

sql_lines = [line for line in sql_path.read_text().splitlines()
             if not line.strip().startswith('--')]
statements = [s.strip() for s in '\n'.join(sql_lines).split(';') if s.strip()]
for statement in statements:
    spark.sql(statement)
print(f'create_tables.sql: {len(statements)} statements executed')

In [ ]:
%run ./initial_dims

## 2. Daily run

Generates facts for `TARGET_DATE` and ingests them into `car_workshop.fact.*`.
For scheduled runs, point a Databricks Job at these two notebooks (or at this one).

In [ ]:
%run ./daily

In [ ]:
%run ./autoloader

## 3. Data quality checks (optional)

NULL audit of every column. `tables_checker.ipynb` is **not** chained here on purpose –
it contains DELETE cells and is meant for manual use.

In [ ]:
%run ./testing/test